In [1]:

# Import necessary libraries
import numpy as np
import pickle
from scipy.optimize import minimize_scalar
from collections import defaultdict
import pandas as pd
import matplotlib.pyplot as plt

# Set random seed for reproducibility
np.random.seed(42)

# Verify available artifacts
import os
print("Available artifacts:")
for f in ['f_canon_rand_primes_N1e7.pkl', 'omega_values_N1e6.pkl', 'omega_values_N1e7.pkl']:
 if os.path.exists(f):
 print(f" ✓ {f}")
 else:
 print(f" ✗ {f} (missing)")


Available artifacts:
 ✓ f_canon_rand_primes_N1e7.pkl
 ✓ omega_values_N1e6.pkl
 ✓ omega_values_N1e7.pkl


In [2]:

# Load the canonical random function prime coefficients
with open('f_canon_rand_primes_N1e7.pkl', 'rb') as f:
 prime_data = pickle.load(f)

primes = prime_data['primes']
a_p = prime_data['a_p']
prime_to_coeff = prime_data['prime_to_coeff']

print(f"Loaded prime data:")
print(f" Number of primes: {len(primes)}")
print(f" Max prime: {primes[-1]}")
print(f" First 5 primes: {primes[:5]}")
print(f" First 5 coefficients: {a_p[:5]}")
print(f" All coefficients have |a_p|=1: {np.allclose(np.abs(a_p), 1.0)}")


KeyError: 'prime_to_coeff'

In [3]:

# Load the canonical random function prime coefficients and check keys
with open('f_canon_rand_primes_N1e7.pkl', 'rb') as f:
 prime_data = pickle.load(f)

print("Keys in prime_data:", prime_data.keys())

primes = prime_data['primes']
a_p = prime_data['a_p']

print(f"\nLoaded prime data:")
print(f" Number of primes: {len(primes)}")
print(f" Max prime: {primes[-1]}")
print(f" First 5 primes: {primes[:5]}")
print(f" First 5 coefficients: {a_p[:5]}")
print(f" All coefficients have |a_p|=1: {np.allclose(np.abs(a_p), 1.0)}")

# Create prime to coefficient dictionary if not present
if 'prime_to_coeff' not in prime_data:
 prime_to_coeff = {p: a_p[i] for i, p in enumerate(primes)}
 print(f"\nCreated prime_to_coeff dictionary with {len(prime_to_coeff)} entries")
else:
 prime_to_coeff = prime_data['prime_to_coeff']


Keys in prime_data: dict_keys(['primes', 'a_p', 'prime_coeff_dict', 'seed', 'max_n', 'generation_method', 'formula'])

Loaded prime data:
 Number of primes: 664579
 Max prime: 9999991
 First 5 primes: [ 2 3 5 7 11]
 First 5 coefficients: [-0.70506063+0.70914702j 0.95243384-0.30474544j -0.11289421-0.99360701j
 -0.81394263-0.58094525j 0.55677833+0.83066112j]
 All coefficients have |a_p|=1: True

Created prime_to_coeff dictionary with 664579 entries


In [4]:

# Actually use the existing dictionary
prime_to_coeff = prime_data['prime_coeff_dict']
print(f"Loaded existing prime_coeff_dict with {len(prime_to_coeff)} entries")

# Verify it matches
print(f"Verification: prime_to_coeff[2] = {prime_to_coeff[2]}, a_p[0] = {a_p[0]}")
print(f"Match: {np.isclose(prime_to_coeff[2], a_p[0])}")


Loaded existing prime_coeff_dict with 664579 entries
Verification: prime_to_coeff[2] = (-0.7050606339757902+0.7091470245426238j), a_p[0] = (-0.7050606339757902+0.7091470245426238j)
Match: True


In [5]:

def prime_factorization(n):
 """Return prime factorization of n as list of primes with multiplicity."""
 factors = []
 d = 2
 while d * d <= n:
 while n % d == 0:
 factors.append(d)
 n //= d
 d += 1
 if n > 1:
 factors.append(n)
 return factors

def compute_f_canon_rand_coeff(n, prime_to_coeff):
 """
 Compute a_n for the canonical random multiplicative function.
 For multiplicative f: a_n = ∏_{p^k || n} a_p^k
 """
 if n == 1:
 return 1.0 + 0j
 
 factors = prime_factorization(n)
 coeff = 1.0 + 0j
 for p in factors:
 if p in prime_to_coeff:
 coeff *= prime_to_coeff[p]
 else:
 # Prime not in our data - should not happen for N <= 10^7
 return 0.0 + 0j
 return coeff

# Test the function
test_cases = [1, 2, 3, 4, 6, 8, 12]
print("Testing coefficient computation:")
for n in test_cases:
 a_n = compute_f_canon_rand_coeff(n, prime_to_coeff)
 print(f" a_{n} = {a_n:.6f}, |a_{n}| = {np.abs(a_n):.6f}")


Testing coefficient computation:
 a_1 = 1.000000+0.000000j, |a_1| = 1.000000
 a_2 = -0.705061+0.709147j, |a_2| = 1.000000
 a_3 = 0.952434-0.304745j, |a_3| = 1.000000
 a_4 = -0.005779-0.999983j, |a_4| = 1.000000
 a_6 = -0.455414+0.890280j, |a_6| = 1.000000
 a_8 = 0.713210+0.700951j, |a_8| = 1.000000
 a_12 = -0.310244-0.950657j, |a_12| = 1.000000


In [6]:

def evaluate_dirichlet_sum_kahan(coeffs, t, omega_values=None):
 """
 Evaluate Dirichlet sum D_F(t; N) = Σ_{n=1}^N a_n / n^(1/2 + it)
 using Kahan compensated summation for high precision.
 
 If omega_values provided, also return omega decomposition.
 """
 N = len(coeffs)
 n_array = np.arange(1, N + 1)
 
 # Compute n^(-1/2 - it) = n^(-1/2) * exp(-it * log(n))
 n_sqrt_inv = 1.0 / np.sqrt(n_array)
 log_n = np.log(n_array)
 phase = np.exp(-1j * t * log_n)
 
 # Terms: a_n / n^(1/2 + it)
 terms = coeffs * n_sqrt_inv * phase
 
 # Kahan summation
 total = 0.0 + 0j
 c = 0.0 + 0j # Running compensation for lost low-order bits
 
 for term in terms:
 y = term - c
 t_new = total + y
 c = (t_new - total) - y
 total = t_new
 
 if omega_values is not None:
 # Compute omega decomposition
 max_omega = int(np.max(omega_values))
 S_k = np.zeros(max_omega + 1, dtype=complex)
 
 for k in range(max_omega + 1):
 mask = (omega_values == k)
 if np.any(mask):
 # Kahan sum for this omega class
 class_total = 0.0 + 0j
 class_c = 0.0 + 0j
 class_terms = terms[mask]
 
 for term in class_terms:
 y = term - class_c
 t_new = class_total + y
 class_c = (t_new - class_total) - y
 class_total = t_new
 
 S_k[k] = class_total
 
 return total, S_k
 
 return total

# Test on a simple case
print("Testing Dirichlet sum evaluation:")
test_coeffs = np.array([compute_f_canon_rand_coeff(n, prime_to_coeff) for n in range(1, 101)])
t_test = 1000.0
D_test = evaluate_dirichlet_sum_kahan(test_coeffs, t_test)
print(f" D_F(t={t_test}, N=100) = {D_test}")
print(f" |D_F| = {np.abs(D_test):.6f}")


Testing Dirichlet sum evaluation:
 D_F(t=1000.0, N=100) = (2.2885532282445795+2.1788143354412233j)
 |D_F| = 3.159859


In [7]:

def evaluate_dirichlet_sum_fast(coeffs, t):
 """
 Fast vectorized evaluation of Dirichlet sum (no Kahan correction).
 Use for coarse grid search to find candidate peaks.
 """
 N = len(coeffs)
 n_array = np.arange(1, N + 1)
 
 n_sqrt_inv = 1.0 / np.sqrt(n_array)
 log_n = np.log(n_array)
 phase = np.exp(-1j * t * log_n)
 
 terms = coeffs * n_sqrt_inv * phase
 return np.sum(terms)

# Test fast vs Kahan
t_test = 1000.0
D_fast = evaluate_dirichlet_sum_fast(test_coeffs, t_test)
D_kahan = evaluate_dirichlet_sum_kahan(test_coeffs, t_test)

print(f"Fast: D_F = {D_fast}, |D_F| = {np.abs(D_fast):.6f}")
print(f"Kahan: D_F = {D_kahan}, |D_F| = {np.abs(D_kahan):.6f}")
print(f"Difference: {np.abs(D_fast - D_kahan):.2e}")


Fast: D_F = (2.2885532282445795+2.1788143354412233j), |D_F| = 3.159859
Kahan: D_F = (2.2885532282445795+2.1788143354412233j), |D_F| = 3.159859
Difference: 0.00e+00


In [8]:

def find_peaks_coarse_grid(coeffs, t_min, t_max, n_grid=10000):
 """
 Phase 1: Coarse grid search using fast vectorized summation.
 Returns grid of t values and magnitudes.
 """
 t_grid = np.linspace(t_min, t_max, n_grid)
 magnitudes = np.zeros(n_grid)
 
 for i, t in enumerate(t_grid):
 D = evaluate_dirichlet_sum_fast(coeffs, t)
 magnitudes[i] = np.abs(D)
 
 if (i + 1) % 2000 == 0:
 print(f" Evaluated {i+1}/{n_grid} grid points...")
 
 return t_grid, magnitudes

def refine_peak_kahan(coeffs, t_initial, search_window=10.0):
 """
 Phase 2: Refine peak location using Kahan summation and scipy optimization.
 Returns refined t and magnitude.
 """
 # Define objective (negative magnitude for minimization)
 def neg_magnitude(t):
 D = evaluate_dirichlet_sum_kahan(coeffs, t)
 return -np.abs(D)
 
 # Use Brent's method for local optimization
 result = minimize_scalar(
 neg_magnitude, 
 bounds=(t_initial - search_window, t_initial + search_window),
 method='bounded'
 )
 
 t_refined = result.x
 mag_refined = -result.fun
 
 return t_refined, mag_refined

print("Peak finding functions defined.")


Peak finding functions defined.


In [9]:

def find_top_peaks_two_phase(coeffs, t_min, t_max, n_peaks=50, n_grid=10000, 
 search_window=10.0, verbose=True):
 """
 Two-phase peak finding:
 1. Coarse grid search with fast summation
 2. Local refinement of top candidates with Kahan summation
 """
 if verbose:
 print(f"\nPhase 1: Coarse grid search over [{t_min}, {t_max}]...")
 
 t_grid, magnitudes = find_peaks_coarse_grid(coeffs, t_min, t_max, n_grid)
 
 # Find local maxima on the grid
 # A point is a local max if it's larger than its neighbors
 is_peak = np.zeros(len(magnitudes), dtype=bool)
 is_peak[1:-1] = (magnitudes[1:-1] > magnitudes[:-2]) & (magnitudes[1:-1] > magnitudes[2:])
 
 peak_indices = np.where(is_peak)[0]
 peak_mags = magnitudes[peak_indices]
 
 if verbose:
 print(f" Found {len(peak_indices)} local maxima on grid")
 
 # Sort by magnitude and take top candidates
 n_candidates = min(n_peaks * 3, len(peak_indices)) # 3x oversampling
 top_candidate_idx = np.argsort(peak_mags)[-n_candidates:][::-1]
 candidate_t = t_grid[peak_indices[top_candidate_idx]]
 
 if verbose:
 print(f"\nPhase 2: Refining top {n_candidates} candidates with Kahan summation...")
 
 refined_peaks = []
 for i, t in enumerate(candidate_t):
 t_refined, mag_refined = refine_peak_kahan(coeffs, t, search_window)
 refined_peaks.append((t_refined, mag_refined))
 
 if verbose and (i + 1) % 20 == 0:
 print(f" Refined {i+1}/{n_candidates} candidates...")
 
 # Sort by refined magnitude and return top n_peaks
 refined_peaks.sort(key=lambda x: x[1], reverse=True)
 refined_peaks = refined_peaks[:n_peaks]
 
 if verbose:
 print(f"\nTop {n_peaks} peaks identified.")
 print(f" Magnitude range: [{refined_peaks[-1][1]:.4f}, {refined_peaks[0][1]:.4f}]")
 
 return refined_peaks

print("Two-phase peak finding function defined.")


Two-phase peak finding function defined.


In [10]:

# Analysis Plan:
print("="*80)
print("ANALYSIS PLAN FOR f_canon_rand")
print("="*80)
print("""
1. DATA PREPARATION (N = 10^4, 10^5, 10^6, 10^7)
 - Compute coefficients a_n for each N using multiplicative property
 - Load pre-computed omega values for N >= 10^6
 - Compute omega values for N = 10^4 and 10^5

2. PEAK FINDING AT EACH N
 - N = 10^4: Direct evaluation on fine grid (single phase, t ∈ [10^4, 2×10^4])
 - N = 10^5: Direct evaluation on fine grid (t ∈ [10^5, 2×10^5])
 - N = 10^6, 10^7: Two-phase approach with coarse grid + Kahan refinement
 - Find top 50 peaks at each N

3. OMEGA DECOMPOSITION AT PEAKS
 - At each peak location, compute S_k for all omega classes
 - Calculate canonical r metric: r = Σ_{j≠k} Re[S_j S̄_k] / Σ_k|S_k|²

4. r-NUMERATOR DECOMPOSITION (N = 10^6, 10^7)
 - Adjacent contribution: Σ_{k} Re[S_{k+1} S̄_k + S_{k-1} S̄_k]
 - Non-adjacent contribution: Σ_{j,k: |j-k|>1} Re[S_j S̄_k]
 - Report fraction of numerator from adjacent pairs

5. COMPARISON WITH ARITHMETIC FUNCTIONS
 - Plot mean r vs N for f_canon_rand
 - Compare to known results for zeta and Liouville
 - Analyze monotonicity and structural changes

Statistical Methods:
- Kahan compensated summation for all final evaluations
- Mean and standard deviation of r across 50 peaks
- Adjacent vs non-adjacent decomposition of r-numerator
""")
print("="*80)


ANALYSIS PLAN FOR f_canon_rand

1. DATA PREPARATION (N = 10^4, 10^5, 10^6, 10^7)
 - Compute coefficients a_n for each N using multiplicative property
 - Load pre-computed omega values for N >= 10^6
 - Compute omega values for N = 10^4 and 10^5

2. PEAK FINDING AT EACH N
 - N = 10^4: Direct evaluation on fine grid (single phase, t ∈ [10^4, 2×10^4])
 - N = 10^5: Direct evaluation on fine grid (t ∈ [10^5, 2×10^5])
 - N = 10^6, 10^7: Two-phase approach with coarse grid + Kahan refinement
 - Find top 50 peaks at each N

3. OMEGA DECOMPOSITION AT PEAKS
 - At each peak location, compute S_k for all omega classes
 - Calculate canonical r metric: r = Σ_{j≠k} Re[S_j S̄_k] / Σ_k|S_k|²

4. r-NUMERATOR DECOMPOSITION (N = 10^6, 10^7)
 - Adjacent contribution: Σ_{k} Re[S_{k+1} S̄_k + S_{k-1} S̄_k]
 - Non-adjacent contribution: Σ_{j,k: |j-k|>1} Re[S_j S̄_k]
 - Report fraction of numerator from adjacent pairs

5. COMPARISON WITH ARITHMETIC FUNCTIONS
 - Plot mean r vs N for f_canon_rand
 - Compare to kn

In [11]:

# Step 1: Prepare data for N = 10^4
print("="*80)
print("STEP 1: DATA PREPARATION")
print("="*80)

N_values = [10**4, 10**5, 10**6, 10**7]
print(f"\nAnalyzing N values: {N_values}")

# N = 10^4: Compute coefficients
print(f"\n[N = 10^4] Computing coefficients...")
N = 10**4
coeffs_1e4 = np.array([compute_f_canon_rand_coeff(n, prime_to_coeff) for n in range(1, N + 1)])
print(f" Computed {len(coeffs_1e4)} coefficients")
print(f" All |a_n| = 1: {np.allclose(np.abs(coeffs_1e4), 1.0)}")
print(f" Sample: a_100 = {coeffs_1e4[99]:.6f}")


STEP 1: DATA PREPARATION

Analyzing N values: [10000, 100000, 1000000, 10000000]

[N = 10^4] Computing coefficients...
 Computed 10000 coefficients
 All |a_n| = 1: True
 Sample: a_100 = 0.229973+0.973197j


In [12]:

# Compute omega values for N = 10^4
print(f"\n[N = 10^4] Computing Omega values...")

def compute_omega_values(N):
 """Compute Omega(n) for n = 1 to N."""
 omega = np.zeros(N, dtype=int)
 
 # Sieve-like approach
 for p in range(2, N + 1):
 if omega[p - 1] == 0: # p is prime (first time seeing it)
 # Mark all multiples of p
 for multiple in range(p, N + 1, p):
 # Count how many times p divides this multiple
 temp = multiple
 while temp % p == 0:
 omega[multiple - 1] += 1
 temp //= p
 
 # Handle n=1 (Omega(1) = 0) - already set
 return omega

omega_1e4 = compute_omega_values(10**4)
print(f" Computed Omega for n=1 to {10**4}")
print(f" Omega(1) = {omega_1e4[0]}")
print(f" Omega(2) = {omega_1e4[1]}")
print(f" Omega(12) = {omega_1e4[11]} (should be 3: 2^2 * 3)")
print(f" Max Omega = {np.max(omega_1e4)}")



[N = 10^4] Computing Omega values...
 Computed Omega for n=1 to 10000
 Omega(1) = 0
 Omega(2) = 1
 Omega(12) = 3 (should be 3: 2^2 * 3)
 Max Omega = 13


In [13]:

# N = 10^5: Compute coefficients and omega values
print(f"\n[N = 10^5] Computing coefficients...")
N = 10**5
coeffs_1e5 = np.array([compute_f_canon_rand_coeff(n, prime_to_coeff) for n in range(1, N + 1)])
print(f" Computed {len(coeffs_1e5)} coefficients")

print(f"\n[N = 10^5] Computing Omega values...")
omega_1e5 = compute_omega_values(10**5)
print(f" Computed Omega for n=1 to {10**5}")
print(f" Max Omega = {np.max(omega_1e5)}")



[N = 10^5] Computing coefficients...


 Computed 100000 coefficients

[N = 10^5] Computing Omega values...
 Computed Omega for n=1 to 100000
 Max Omega = 16


In [14]:

# N = 10^6: Compute coefficients and load omega values
print(f"\n[N = 10^6] Computing coefficients...")
N = 10**6

# This is large - compute in chunks to monitor progress
coeffs_1e6 = np.zeros(N, dtype=complex)
chunk_size = 100000

for i in range(0, N, chunk_size):
 end = min(i + chunk_size, N)
 for j in range(i, end):
 coeffs_1e6[j] = compute_f_canon_rand_coeff(j + 1, prime_to_coeff)
 print(f" Computed coefficients {i+1} to {end}...")

print(f" Total: {len(coeffs_1e6)} coefficients")
print(f" All |a_n| = 1: {np.allclose(np.abs(coeffs_1e6), 1.0)}")

# Load pre-computed omega values
print(f"\n[N = 10^6] Loading pre-computed Omega values...")
with open('omega_values_N1e6.pkl', 'rb') as f:
 omega_1e6 = pickle.load(f)
print(f" Loaded Omega for n=1 to {len(omega_1e6)}")
print(f" Max Omega = {np.max(omega_1e6)}")



[N = 10^6] Computing coefficients...


 Computed coefficients 1 to 100000...


 Computed coefficients 100001 to 200000...


 Computed coefficients 200001 to 300000...


 Computed coefficients 300001 to 400000...


...

 Computed coefficients 500001 to 600000...


 Computed coefficients 600001 to 700000...


 Computed coefficients 700001 to 800000...


 Computed coefficients 800001 to 900000...


 Computed coefficients 900001 to 1000000...
 Total: 1000000 coefficients
 All |a_n| = 1: True

[N = 10^6] Loading pre-computed Omega values...
 Loaded Omega for n=1 to 1000000
 Max Omega = 19


In [15]:

# N = 10^7: This will take significant time
# Compute coefficients in larger chunks
print(f"\n[N = 10^7] Computing coefficients (this will take several minutes)...")
N = 10**7

coeffs_1e7 = np.zeros(N, dtype=complex)
chunk_size = 1000000 # 1M at a time

import time
start_time = time.time()

for i in range(0, N, chunk_size):
 chunk_start = time.time()
 end = min(i + chunk_size, N)
 for j in range(i, end):
 coeffs_1e7[j] = compute_f_canon_rand_coeff(j + 1, prime_to_coeff)
 chunk_time = time.time() - chunk_start
 print(f" Computed coefficients {i+1} to {end} ({chunk_time:.1f}s)...")

total_time = time.time() - start_time
print(f" Total: {len(coeffs_1e7)} coefficients in {total_time:.1f}s")
print(f" All |a_n| = 1: {np.allclose(np.abs(coeffs_1e7), 1.0)}")



[N = 10^7] Computing coefficients (this will take several minutes)...


 Computed coefficients 1 to 1000000 (10.2s)...


 Computed coefficients 1000001 to 2000000 (17.4s)...


 Computed coefficients 2000001 to 3000000 (21.9s)...


 Computed coefficients 3000001 to 4000000 (25.3s)...


...

 Computed coefficients 5000001 to 6000000 (30.7s)...


 Computed coefficients 6000001 to 7000000 (33.3s)...


 Computed coefficients 7000001 to 8000000 (35.5s)...


 Computed coefficients 8000001 to 9000000 (37.6s)...


 Computed coefficients 9000001 to 10000000 (39.5s)...
 Total: 10000000 coefficients in 279.4s
 All |a_n| = 1: True


In [16]:

# Load pre-computed omega values for N = 10^7
print(f"\n[N = 10^7] Loading pre-computed Omega values...")
with open('omega_values_N1e7.pkl', 'rb') as f:
 omega_1e7 = pickle.load(f)
print(f" Loaded Omega for n=1 to {len(omega_1e7)}")
print(f" Max Omega = {np.max(omega_1e7)}")

print("\n" + "="*80)
print("DATA PREPARATION COMPLETE")
print("="*80)
print(f"Ready to analyze at N ∈ {{10^4, 10^5, 10^6, 10^7}}")



[N = 10^7] Loading pre-computed Omega values...
 Loaded Omega for n=1 to 10000000
 Max Omega = 23

DATA PREPARATION COMPLETE
Ready to analyze at N ∈ {10^4, 10^5, 10^6, 10^7}


In [17]:

# Step 2: Find peaks at N = 10^4
print("="*80)
print("STEP 2: PEAK FINDING")
print("="*80)

print(f"\n[N = 10^4] Finding top 50 peaks in t ∈ [10^4, 2×10^4]...")
print("Using direct evaluation with fine grid (single phase)...")

t_min, t_max = 10**4, 2 * 10**4
n_grid = 20000 # Fine grid for direct search

t_grid, mags = find_peaks_coarse_grid(coeffs_1e4, t_min, t_max, n_grid)

# Find local maxima
is_peak = np.zeros(len(mags), dtype=bool)
is_peak[1:-1] = (mags[1:-1] > mags[:-2]) & (mags[1:-1] > mags[2:])

peak_indices = np.where(is_peak)[0]
peak_mags = mags[peak_indices]

print(f" Found {len(peak_indices)} local maxima")

# Get top 50
top_50_idx = np.argsort(peak_mags)[-50:][::-1]
peaks_1e4 = [(t_grid[peak_indices[i]], peak_mags[i]) for i in top_50_idx]

print(f"\nTop 50 peaks identified:")
print(f" Magnitude range: [{peaks_1e4[-1][1]:.4f}, {peaks_1e4[0][1]:.4f}]")
print(f" Mean magnitude: {np.mean([p[1] for p in peaks_1e4]):.4f}")


STEP 2: PEAK FINDING

[N = 10^4] Finding top 50 peaks in t ∈ [10^4, 2×10^4]...
Using direct evaluation with fine grid (single phase)...


 Evaluated 2000/20000 grid points...


 Evaluated 4000/20000 grid points...


 Evaluated 6000/20000 grid points...


 Evaluated 8000/20000 grid points...


...

 Evaluated 12000/20000 grid points...


 Evaluated 14000/20000 grid points...


 Evaluated 16000/20000 grid points...


 Evaluated 18000/20000 grid points...


 Evaluated 20000/20000 grid points...
 Found 5798 local maxima

Top 50 peaks identified:
 Magnitude range: [16.8727, 37.3612]
 Mean magnitude: 21.5844


In [18]:

# Find peaks at N = 10^5
print(f"\n[N = 10^5] Finding top 50 peaks in t ∈ [10^5, 2×10^5]...")
print("Using direct evaluation with fine grid (single phase)...")

t_min, t_max = 10**5, 2 * 10**5
n_grid = 20000

t_grid, mags = find_peaks_coarse_grid(coeffs_1e5, t_min, t_max, n_grid)

# Find local maxima
is_peak = np.zeros(len(mags), dtype=bool)
is_peak[1:-1] = (mags[1:-1] > mags[:-2]) & (mags[1:-1] > mags[2:])

peak_indices = np.where(is_peak)[0]
peak_mags = mags[peak_indices]

print(f" Found {len(peak_indices)} local maxima")

# Get top 50
top_50_idx = np.argsort(peak_mags)[-50:][::-1]
peaks_1e5 = [(t_grid[peak_indices[i]], peak_mags[i]) for i in top_50_idx]

print(f"\nTop 50 peaks identified:")
print(f" Magnitude range: [{peaks_1e5[-1][1]:.4f}, {peaks_1e5[0][1]:.4f}]")
print(f" Mean magnitude: {np.mean([p[1] for p in peaks_1e5]):.4f}")



[N = 10^5] Finding top 50 peaks in t ∈ [10^5, 2×10^5]...
Using direct evaluation with fine grid (single phase)...


 Evaluated 2000/20000 grid points...


 Evaluated 4000/20000 grid points...


 Evaluated 6000/20000 grid points...


 Evaluated 8000/20000 grid points...


...

 Evaluated 12000/20000 grid points...


 Evaluated 14000/20000 grid points...


 Evaluated 16000/20000 grid points...


 Evaluated 18000/20000 grid points...


 Evaluated 20000/20000 grid points...
 Found 7259 local maxima

Top 50 peaks identified:
 Magnitude range: [20.7049, 41.9566]
 Mean magnitude: 25.6646


In [19]:

# Find peaks at N = 10^6 using two-phase approach
print(f"\n[N = 10^6] Finding top 50 peaks in t ∈ [10^6, 2×10^6]...")
print("Using two-phase approach (coarse grid + Kahan refinement)...")

t_min, t_max = 10**6, 2 * 10**6

import time
start_time = time.time()

peaks_1e6 = find_top_peaks_two_phase(
 coeffs_1e6, 
 t_min, 
 t_max, 
 n_peaks=50, 
 n_grid=10000,
 search_window=50.0,
 verbose=True
)

elapsed = time.time() - start_time
print(f"\nCompleted in {elapsed:.1f}s")
print(f" Magnitude range: [{peaks_1e6[-1][1]:.4f}, {peaks_1e6[0][1]:.4f}]")
print(f" Mean magnitude: {np.mean([p[1] for p in peaks_1e6]):.4f}")



[N = 10^6] Finding top 50 peaks in t ∈ [10^6, 2×10^6]...
Using two-phase approach (coarse grid + Kahan refinement)...

Phase 1: Coarse grid search over [1000000, 2000000]...


 Evaluated 2000/10000 grid points...


 Evaluated 4000/10000 grid points...


 Evaluated 6000/10000 grid points...


 Evaluated 8000/10000 grid points...


...

 Refined 80/150 candidates...


 Refined 100/150 candidates...


 Refined 120/150 candidates...


 Refined 140/150 candidates...



Top 50 peaks identified.
 Magnitude range: [11.1849, 49.9124]

Completed in 1142.3s
 Magnitude range: [11.1849, 49.9124]
 Mean magnitude: 19.3995


In [20]:

# Find peaks at N = 10^7 using two-phase approach
print(f"\n[N = 10^7] Finding top 50 peaks in t ∈ [10^7, 2×10^7]...")
print("Using two-phase approach (coarse grid + Kahan refinement)...")
print("Note: This will take significant time due to large N...")

t_min, t_max = 10**7, 2 * 10**7

start_time = time.time()

peaks_1e7 = find_top_peaks_two_phase(
 coeffs_1e7, 
 t_min, 
 t_max, 
 n_peaks=50, 
 n_grid=10000,
 search_window=100.0,
 verbose=True
)

elapsed = time.time() - start_time
print(f"\nCompleted in {elapsed:.1f}s ({elapsed/60:.1f} min)")
print(f" Magnitude range: [{peaks_1e7[-1][1]:.4f}, {peaks_1e7[0][1]:.4f}]")
print(f" Mean magnitude: {np.mean([p[1] for p in peaks_1e7]):.4f}")


TimeoutError: Code execution timed out after 1200 seconds

In [21]:

# The full N=10^7 search timed out. Let's use a coarser grid and fewer refinements
print(f"\n[N = 10^7] Finding top 50 peaks in t ∈ [10^7, 2×10^7]...")
print("Using optimized two-phase approach (coarser grid, reduced refinements)...")

t_min, t_max = 10**7, 2 * 10**7

start_time = time.time()

# Phase 1: Coarser grid
print(f"\nPhase 1: Coarse grid search over [{t_min}, {t_max}]...")
n_grid = 5000 # Reduced from 10000
t_grid = np.linspace(t_min, t_max, n_grid)
magnitudes = np.zeros(n_grid)

for i, t in enumerate(t_grid):
 D = evaluate_dirichlet_sum_fast(coeffs_1e7, t)
 magnitudes[i] = np.abs(D)
 
 if (i + 1) % 1000 == 0:
 elapsed = time.time() - start_time
 print(f" Evaluated {i+1}/{n_grid} grid points ({elapsed:.1f}s)...")

print(f"Phase 1 completed in {time.time() - start_time:.1f}s")


TimeoutError: Code execution timed out after 1200 seconds

In [22]:

# Even the coarse grid is too slow. Let's optimize the evaluation function itself
# by pre-computing all the invariant parts

print("Optimizing evaluation for N=10^7...")
print("Pre-computing n^(-1/2) and log(n)...")

N = 10**7
n_array = np.arange(1, N + 1, dtype=np.float64)
n_sqrt_inv = 1.0 / np.sqrt(n_array)
log_n = np.log(n_array)

print(f"Pre-computed arrays of length {len(n_array)}")
print(f"Memory usage: ~{(n_sqrt_inv.nbytes + log_n.nbytes) / 1e9:.2f} GB")

def evaluate_dirichlet_sum_optimized(t):
 """Ultra-fast evaluation using pre-computed arrays."""
 phase = np.exp(-1j * t * log_n)
 terms = coeffs_1e7 * n_sqrt_inv * phase
 return np.sum(terms)

# Test speed
import time
t_test = 10**7
start = time.time()
D_test = evaluate_dirichlet_sum_optimized(t_test)
elapsed = time.time() - start
print(f"\nTest evaluation at t={t_test}: |D| = {np.abs(D_test):.4f}")
print(f"Time per evaluation: {elapsed:.3f}s")
print(f"Estimated time for 5000 grid points: {elapsed * 5000 / 60:.1f} min")


TimeoutError: Code execution timed out after 673 seconds